# LangChain Expression Language

In [ ]:
%pip install langchain langchain-openai langchain-community langchain-text-splitters --upgrade

In [ ]:
import getpass
import os

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

In [ ]:
from operator import itemgetter
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnablePassthrough

------------------------------------------

## Branching Out with _Conditional Logic:_
The runnable is initialized with a list of (condition, runnable) pairs and a default branch.

When operating on an input, the first condition that evaluates to True is selected, and the corresponding runnable is run on the input.

If no condition evaluates to True, the default branch is run on the input.

In [ ]:
branch = RunnableBranch(
                (lambda x: x == 'hello', lambda x: x),
                (lambda x: isinstance(x, str), lambda x: x.upper()),
                (lambda x: "This is the default case, in case no above lambda functions match."), #generic, catch-all case
            )

print(branch.invoke("hello")) # "hello"
print(branch.invoke(None)) # "This is the default case"

hello
This is the default case, in case no above lambda functions match.


---

## Branching and _Merging_

```
     Input
      / \
     /   \
 Branch1 Branch2
     \   /
      \ /
    Combine

```

### First Example:

In [ ]:
planner = (
    ChatPromptTemplate.from_template("Generate an argument about: {input}")
    | ChatOpenAI()
    | StrOutputParser()
    | {"base_response": RunnablePassthrough()} # output of the String Output Parser (from the RunnablePassthrough)
    # was associated with the 'base_response' key in the dictionary
)

#two prompt chains: arguments_for and arguments_against
arguments_for = (
    ChatPromptTemplate.from_template(
        "List the pros or positive aspects of {base_response}"
    )
    | ChatOpenAI()
    | StrOutputParser()
)

arguments_against = (
    ChatPromptTemplate.from_template(
        "List the cons or negative aspects of {base_response}"
    )
    | ChatOpenAI()
    | StrOutputParser()
)

final_responder = (
    ChatPromptTemplate.from_messages(
        [
            ("ai", "{original_response}"),
            ("human", "Pros:\n{results_1}\n\nCons:\n{results_2}"),
            ("system", "Generate a final response given the critique"),
        ]
    )
    | ChatOpenAI()
    | StrOutputParser()
)

chain = (
    planner
    | {
        "results_1": arguments_for,
        "results_2": arguments_against,
        "original_response": itemgetter("base_response"),
    }
    | final_responder
)

the below output will take some time because of the synchronous API calls happening there
(synchronously waiting for branch 1 and branch 2 to finish before taking the outputs of both branches and doing another LLM API request)

In [ ]:
chain.invoke({"input": "scrum"})

"While Scrum offers numerous benefits such as flexibility, collaboration, continuous improvement, and teamwork, it is important to acknowledge some potential drawbacks. These include the learning curve associated with adapting to a new framework, the need for a structured approach that may not align with all project requirements, the time commitment required for regular meetings and communication, potential challenges with team collaboration, limited flexibility with fixed deadlines, and reduced predictability in project timelines and outcomes.\n\nIt's essential for organizations to carefully assess their project requirements, team dynamics, and stakeholder expectations before committing to Scrum as a project management framework. While Scrum can bring about significant advantages in the right context, there may be instances where a more traditional or structured approach is better suited for achieving project goals. Ultimately, choosing the right project management framework requires 

### Example Two:

In [ ]:
joke_chain = (
    ChatPromptTemplate.from_template("Tell me a joke about {topic}")
    | ChatOpenAI()
    | StrOutputParser()
    | {"joke": RunnablePassthrough(), "topic": RunnablePassthrough()}
)

explain_joke = (
    ChatPromptTemplate.from_template("Explain the joke: {joke}")
    | ChatOpenAI()
    | StrOutputParser()
)

benefits_of_joke = (
    ChatPromptTemplate.from_template("List the benefits of this joke: {joke}")
    | ChatOpenAI()
    | StrOutputParser()
)

final_responder = (
    ChatPromptTemplate.from_messages(
        [
            ("system", "You are responsible for generating a small analysis of a joke. The topic will be: {topic}"),
            ("ai", "{joke}. The benefits of this joke are: {benefits}"),
            ("human", "The explanation of the joke is: {explanation}"),
            ("human", "Generate a small analysis of the joke. Analysis: "),
        ]
    )
    | ChatOpenAI()
    | StrOutputParser()
)

final_chain = (
    {"topic": RunnablePassthrough()}
    | joke_chain
    | { #reattaching the joke and the topic keys because
       # we need them to get passed to the final responder
        "explanation": explain_joke,
        "benefits": benefits_of_joke,
        "joke": itemgetter("joke"),
        "topic": itemgetter("topic"),
    }
    | final_responder
)

final_chain.invoke({"topic": "bears"})

'The joke "Why did the bear break up with her boyfriend? Because he was always grizzly!" relies on wordplay and a clever twist on the stereotypical behavior of bears. By using the term "grizzly" to describe both the bear and her boyfriend, the joke cleverly sets up a pun that plays on the dual meanings of the word. This creates a humorous and unexpected punchline that elicits a chuckle from the audience. Additionally, the joke humanizes the bear by attributing relatable relationship issues to her, adding another layer of whimsy to the humor. Overall, the joke is a lighthearted and witty play on words that entertains through its clever use of language and clever twist on animal behavior.'